In [88]:
import numpy as np
import pandas as pd

In [4]:
# read in dataset
df_hs = pd.read_csv("data_use.csv")
df_hs

,user.index,date,time,5min_datetime,steps,rolling_sum,rt,sugg.decision.5min,send.sedentary,availability
0,1,7/22/2015,9:05:00,NaN,0,0,1,NaN,0,1
1,1,7/22/2015,9:10:00,NaN,0,0,1,NaN,0,1
2,1,7/22/2015,9:15:00,NaN,0,0,1,NaN,0,1
3,1,7/22/2015,9:20:00,NaN,0,0,1,NaN,0,1
4,1,7/22/2015,9:25:00,NaN,0,0,1,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...
228235,37,1/26/2016,20:40:00,NaN,0,0,1,NaN,0,1
228236,37,1/26/2016,20:45:00,NaN,0,0,1,NaN,0,1
228237,37,1/26/2016,20:50:00,NaN,0,0,1,NaN,0,1
228238,37,1/26/2016,20:55:00,NaN,0,0,1,NaN,0,1


In [ ]:
# create subset of the data to work on


## Define `transform_dnn`

In [107]:
def transform_dnn(df, day_partitions = 4, day_measure = 144, units = 37):
    """
    Transfom df into shape suitable for DNN use case.

    Parameters
    ----------
    df : pandas DataFrame
         a matrix of size Z x 10 (with a set of properties to be defined)
         ...
    day_partitions : int
        The number of partitions to split an individual day into. The 
        default value is 4. Maximally, the value of day_partitions is 
        day_measure.
    day_measure : int
        The number of measurements per "day" in df. The default is 144,
        which aligns with the measurements per day in the HeartSteps dataset.
    units : int
        The number of units in df. Default is 37 (in line with the HeartSteps dataset)        
    Returns
    ----------
    tuple (M, A)
    where M is a units x (day_partitions * D) x (day_measure / day_partitions) np array 
    that contains the outcome vectors of interest and A is a np array shape units x (day_partitions * D)
    that indicates in which unit/time combination a treatment was observed.
    D is the maximum length of days observed among all unit trials.
    """
    # first, find the maximum trial length in days
    # units are 1-indexed
    max_days = 0
    num_days = []
    for i in range(1, units + 1):
        len_i = len(df[df['user.index'] == i]) // day_measure
        num_days.append(len_i)
        if len_i > max_days:
            max_days = len_i
    # init M and A
    M = np.zeros((units, day_partitions * max_days, day_measure // day_partitions))
    A = np.zeros((units, day_partitions * max_days))
    
    for i in range(units):
        # note: i is used to index into M and A.
        # unit_ind is used to retrieve unit specific values from df
        unit_ind = i + 1
        
        # slice by unit
        df_unit = df.loc[df["user.index"] == unit_ind]
        
        # pad first to max trial length to maintain partition size across units
        trial_steps = df_unit["steps"].to_numpy()
        trial_trt = df_unit["treatment"].to_numpy()
        if num_days[i] < max_days:
            # pad with 0s, step values are masked by trt
            pad = np.zeros((max_days - num_days[i]) * day_measure)
            trial_steps = np.append(trial_steps, pad)
            pad_trt = np.zeros((max_days - num_days[i]) * day_measure)
            trial_trt = np.append(trial_trt, pad_trt)
            
        # first split into days, then into partitions
        lst_days = np.split(trial_steps, max_days)
        parts = np.array([np.split(x, day_partitions) for x in lst_days])
         
        # Shape of parts is max_days x day_partitions x (day_measure // day_partitions)
        # Flatten along d dimension to achieve (day_partitions * max_days) x (day_measure // day_partitions) shape
        unit_parts = parts.reshape((max_days * day_partitions, day_measure // day_partitions))
        
        # Repeat for treatment col. 
        lst_days_trt = np.split(trial_trt, max_days)
        parts_trt = np.array([np.split(x, day_partitions) for x in lst_days_trt])
        unit_trt = parts_trt.reshape((max_days * day_partitions, day_measure // day_partitions))
        
        # if any of the trt values within the partition is 1, set the value in the final array to 1
        unit_trt_agg = unit_trt.any(axis = 1).astype(int)
        
        M[i] = unit_parts
        A[i] = unit_trt_agg
    return M, A
    

## Testing `transform_dnn` with symmetrical play data

We create a small test dataset as a proof of concept of the above data transformation. This data set will have 3 units over a time span of 4 days, with 12 measurements a day. We partition each day into 3 segments (4 measurements per partition). The treatment is randomly applied across each day and time span. This is a simpler example than the actual use case, as the number of days within the trial is consistent for all users. See the section below for a more complex example.

In [90]:
# create test set

# user inds
u1 = np.repeat(1, 12 * 4)
u2 = np.repeat(2, 12 * 4)
u3 = np.repeat(3, 12 * 4)
user_index = np.concatenate((u1, u2, u3))

# step measurements (for simplicity, between 1 and 10)
rng = np.random.default_rng()
u1_steps = rng.integers(low=1, high=10, size=12 * 4)
u2_steps = rng.integers(low=1, high=10, size=12 * 4)
u3_steps = rng.integers(low=0, high=10, size=12 * 4)
steps = np.concatenate((u1_steps, u2_steps, u3_steps))
 
# treatment measurements
# in practice, it seems much more likely to see no treatment rather than a treatment
u1_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u2_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u3_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
trt = np.concatenate((u1_trt, u2_trt, u3_trt))

# create final dataframe
final_np = np.stack((user_index, steps, trt), axis = 1)
df_symtest = pd.DataFrame(final_np, columns = ["user.index", "steps", "treatment"])
df_symtest

,user.index,steps,treatment
0,1,2,0
1,1,4,0
2,1,7,0
3,1,2,0
4,1,1,0
...,...,...,...
139,3,4,0
140,3,0,0
141,3,8,0
142,3,3,0


Now that we have created our test set, we can use `transform_dnn` to demonstrate the desired transformation properties.

In [91]:
M, A = transform_dnn(df_symtest, day_partitions = 3, day_measure = 12, units = 3)

In [92]:
M

array([[[2., 4., 7., 2.],
        [1., 9., 8., 8.],
        [5., 6., 8., 4.],
        [4., 8., 3., 9.],
        [9., 7., 7., 4.],
        [1., 4., 9., 2.],
        [4., 8., 5., 7.],
        [7., 5., 9., 4.],
        [5., 9., 6., 6.],
        [6., 7., 4., 2.],
        [8., 2., 2., 4.],
        [5., 7., 6., 8.]],

       [[6., 3., 8., 5.],
        [9., 4., 8., 4.],
        [5., 7., 3., 9.],
        [5., 2., 7., 1.],
        [5., 6., 2., 9.],
        [5., 6., 5., 5.],
        [3., 7., 5., 2.],
        [1., 7., 3., 1.],
        [7., 5., 9., 6.],
        [7., 7., 7., 5.],
        [3., 1., 8., 1.],
        [1., 9., 5., 2.]],

       [[3., 3., 9., 9.],
        [6., 8., 0., 0.],
        [0., 7., 5., 5.],
        [3., 3., 5., 3.],
        [2., 9., 2., 9.],
        [3., 2., 8., 2.],
        [4., 4., 9., 8.],
        [3., 1., 4., 9.],
        [7., 6., 5., 1.],
        [7., 2., 5., 3.],
        [6., 2., 6., 4.],
        [0., 8., 3., 7.]]])

In [93]:
A

array([[0., 0., 0., 1., 0., 0., 1., 1., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0.],
       [1., 1., 1., 0., 1., 0., 0., 1., 0., 1., 0., 0.]])

## Testing `transform_dnn` on a asymmetrical dataset

Next, we test the function on a dataset with asymmetrical trial periods. As before, we will have 3 units, but this time, one of the units has a longer 6 day trial period. The remaining units have a 4 day trial period. There are still 12 measurements a day, and each day has 3 segments.

In [105]:
# create test set
# user inds, u1 has 6 day trial period
u1 = np.repeat(1, 12 * 6)
u2 = np.repeat(2, 12 * 4)
u3 = np.repeat(3, 12 * 4)
user_index = np.concatenate((u1, u2, u3))

# step measurements (for simplicity, between 1 and 10)
rng = np.random.default_rng()
u1_steps = rng.integers(low=1, high=10, size=12 * 6)
u2_steps = rng.integers(low=1, high=10, size=12 * 4)
u3_steps = rng.integers(low=0, high=10, size=12 * 4)
steps = np.concatenate((u1_steps, u2_steps, u3_steps))
 
# treatment measurements
# in practice, it seems much more likely to see no treatment rather than a treatment
u1_trt = np.random.choice([0, 1], 12 * 6, p=[0.8, 0.2])
u2_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u3_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
trt = np.concatenate((u1_trt, u2_trt, u3_trt))

# create final dataframe
final_np = np.stack((user_index, steps, trt), axis = 1)
df_asymtest = pd.DataFrame(final_np, columns = ["user.index", "steps", "treatment"])
df_asymtest

,user.index,steps,treatment
0,1,6,0
1,1,2,0
2,1,6,1
3,1,6,0
4,1,4,1
...,...,...,...
163,3,9,0
164,3,1,0
165,3,3,0
166,3,6,0


In [108]:
M_a, A_a = transform_dnn(df_asymtest, day_partitions = 3, day_measure = 12, units = 3)

In [109]:
M_a

array([[[6., 2., 6., 6.],
        [4., 5., 4., 2.],
        [3., 1., 7., 7.],
        [5., 2., 9., 7.],
        [4., 2., 6., 2.],
        [8., 4., 4., 3.],
        [7., 9., 9., 4.],
        [1., 6., 3., 8.],
        [4., 5., 7., 1.],
        [2., 1., 6., 7.],
        [7., 6., 9., 1.],
        [3., 9., 1., 8.],
        [5., 1., 2., 6.],
        [8., 3., 2., 6.],
        [5., 8., 3., 1.],
        [9., 6., 2., 4.],
        [7., 6., 8., 3.],
        [5., 2., 2., 9.]],

       [[6., 2., 5., 2.],
        [2., 7., 2., 3.],
        [7., 4., 8., 5.],
        [3., 6., 3., 5.],
        [3., 8., 9., 2.],
        [5., 6., 4., 1.],
        [8., 5., 4., 3.],
        [9., 1., 1., 8.],
        [4., 9., 6., 5.],
        [1., 1., 4., 1.],
        [2., 6., 3., 7.],
        [9., 7., 3., 4.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]],

       [[6., 9., 1., 2.],
        [2., 2., 8., 3.],
        

In [110]:
A_a

array([[1., 1., 1., 1., 1., 0., 1., 0., 0., 1., 0., 1., 1., 1., 0., 1.,
        1., 1.],
       [0., 1., 0., 0., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0.,
        0., 0.],
       [1., 0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
        0., 0.]])